# **Classification Model Training Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"


In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# No additional packages are required for this notebook.


### 0.b Import Packages

In [ ]:
import numpy as np
import pandas as pd
import altair as alt

from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

alt.data_transformers.disable_max_rows()


---
## B. Business Understanding

In [ ]:
business_use_case_description = """
The goal of this project is to predict whether a customer will place another order within 90 days.
A reliable re-order classifier helps the retail business identify which customers are most likely to buy again,
so the team can plan targeted marketing, replenishment, and customer-retention actions more effectively.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [ ]:
business_objectives = """
The main objective is to improve customer retention and campaign efficiency.
Accurate predictions help the business focus incentives and follow-up actions on customers with a high likelihood of re-ordering,
while inaccurate predictions can waste promotional budget, miss repeat-sales opportunities, and reduce trust in the analytics process.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [ ]:
stakeholders_expectations_explanations = """
The key stakeholders are marketing, sales, customer-retention, and operations teams.
They expect a model that is simple to explain, reproducible, and strong enough to rank customers by re-order likelihood.
The outputs can support targeted outreach, prioritised follow-up lists, and better planning of stock and campaign timing.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Datasets


In [ ]:
data_path = at.folder_path

X_train = pd.read_csv(data_path / 'X_train.csv')
y_train = pd.read_csv(data_path / 'y_train.csv')

X_val = pd.read_csv(data_path / 'X_val.csv')
y_val = pd.read_csv(data_path / 'y_val.csv')

X_test = pd.read_csv(data_path / 'X_test.csv')
y_test = pd.read_csv(data_path / 'y_test.csv')

print('Loaded datasets successfully:')
for split_name, X_split, y_split in [
    ('train', X_train, y_train),
    ('validation', X_val, y_val),
    ('test', X_test, y_test),
]:
    print(f"  {split_name:<10} X={X_split.shape}  y={y_split.shape}")

display(X_train.head())
display(y_train.head())


### C.2 Define Target variable

In [ ]:
target_name = y_train.columns[0]
print(f"Target column detected: {target_name}")
print('Target column exists in all splits:', all(target_name in df.columns for df in [y_train, y_val, y_test]))


In [ ]:
target_quality_checks = pd.DataFrame([
    {
        'split': split_name,
        'rows': len(y_split),
        'unique_values': sorted(pd.Series(y_split[target_name]).dropna().unique().tolist()),
        'positive_rate': float(pd.Series(y_split[target_name]).mean()),
        'missing_values': int(pd.Series(y_split[target_name]).isna().sum()),
    }
    for split_name, y_split in [('train', y_train), ('validation', y_val), ('test', y_test)]
])

display(target_quality_checks)


In [ ]:
target_definition_explanations = f"""
The target variable is `{target_name}` and represents whether a customer re-orders within 90 days.
This target directly matches the business problem because it converts repeat-purchase behaviour into a clear binary outcome.
The class balance is stable across the three splits, with positive rates between {target_quality_checks['positive_rate'].min():.2%} and {target_quality_checks['positive_rate'].max():.2%},
which is suitable for supervised classification while still signalling moderate class imbalance.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [ ]:
for y_split in [y_train, y_val, y_test]:
    y_split[target_name] = pd.to_numeric(y_split[target_name], errors='coerce').fillna(0).astype(int)

train_df = X_train.copy()
train_df[target_name] = y_train[target_name].values

val_df = X_val.copy()
val_df[target_name] = y_val[target_name].values

test_df = X_test.copy()
test_df[target_name] = y_test[target_name].values

print('Created modelling dataframes with predictors + target:')
for split_name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f"  {split_name:<10} {df.shape}")


### C.4 Explore Target variable

In [ ]:
target_distribution = pd.DataFrame([
    {
        'split': split_name,
        'class': class_value,
        'count': int((df[target_name] == class_value).sum()),
        'percentage': float((df[target_name] == class_value).mean()),
    }
    for split_name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]
    for class_value in [0, 1]
])

display(target_distribution)

alt.Chart(target_distribution).mark_bar().encode(
    x=alt.X('split:N', title='Dataset split'),
    y=alt.Y('count:Q', title='Number of customers'),
    color=alt.Color('class:N', title='Target class'),
    xOffset='class:N',
    tooltip=['split', 'class', 'count', alt.Tooltip('percentage:Q', format='.2%')]
).properties(
    title='Target distribution by data split',
    width=500,
    height=320,
)


In [ ]:
target_distribution_pivot = target_distribution.pivot(index='split', columns='class', values='percentage').rename(columns={0: 'negative_rate', 1: 'positive_rate'})
display(target_distribution_pivot)


In [ ]:
target_distribution_explanations = f"""
The target is moderately imbalanced: the positive class represents roughly {target_distribution[target_distribution['class'] == 1]['percentage'].mean():.2%} of customers.
This means accuracy alone would be misleading because a naive model could predict mostly zeros and still appear strong.
For that reason, the later evaluation focuses on recall, precision, F1-score, and ROC-AUC in addition to accuracy.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `\<put feature name here\>`

In [ ]:
feature_1_name = 'n_orders'
feature_1_summary = train_df.groupby(feature_1_name)[target_name].agg(
    customers='size',
    reorder_rate='mean'
).reset_index()

display(feature_1_summary)

alt.Chart(feature_1_summary).mark_bar().encode(
    x=alt.X(f'{feature_1_name}:O', title='Number of historical orders'),
    y=alt.Y('reorder_rate:Q', title='Re-order rate'),
    tooltip=[feature_1_name, 'customers', alt.Tooltip('reorder_rate:Q', format='.2%')]
).properties(
    title='Re-order rate by historical order count (train split)',
    width=500,
    height=320,
)


In [ ]:
display(train_df.groupby(target_name)[feature_1_name].describe().round(2))


In [ ]:
feature_1_insights = f"""
`{feature_1_name}` is a strong behavioural feature because it measures how often a customer has purchased historically.
In the training split, customers with the highest order counts show the highest re-order rates, which is consistent with the business expectation that engaged customers are more likely to return.
The feature is also easy to explain to stakeholders and has no missing values in the provided modelling data.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `\<put feature name here\>`

In [ ]:
feature_2_name = 'total_spend'
train_total_spend = train_df[[feature_2_name, target_name]].copy()
train_total_spend['spend_band'] = pd.qcut(train_total_spend[feature_2_name], q=10, duplicates='drop')
feature_2_summary = train_total_spend.groupby('spend_band').agg(
    customers=(target_name, 'size'),
    reorder_rate=(target_name, 'mean'),
    min_spend=(feature_2_name, 'min'),
    max_spend=(feature_2_name, 'max')
).reset_index(drop=True)

display(feature_2_summary)


In [ ]:
feature_2_insights = f"""
`{feature_2_name}` captures customer monetary value and separates low-value from high-value buyers.
Customers in the higher spend bands generally show stronger re-order rates, suggesting that historical spend carries predictive signal for future purchases.
The variable is continuous and right-skewed, so later data-preparation steps focus on outlier handling and scale-friendly transformations.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.6 Explore Feature of Interest `\<put feature name here\>`


In [ ]:
feature_n_name = 'avg_order_value'
train_avg_order_value = train_df[[feature_n_name, target_name]].copy()
train_avg_order_value['value_band'] = pd.qcut(train_avg_order_value[feature_n_name], q=10, duplicates='drop')
feature_n_summary = train_avg_order_value.groupby('value_band').agg(
    customers=(target_name, 'size'),
    reorder_rate=(target_name, 'mean'),
    min_value=(feature_n_name, 'min'),
    max_value=(feature_n_name, 'max')
).reset_index(drop=True)

display(feature_n_summary)


In [ ]:
numeric_signal_summary = train_df[['n_orders', 'total_spend', 'avg_order_value', 'max_order_value', target_name]].corr(numeric_only=True)[[target_name]].sort_values(target_name, ascending=False)
display(numeric_signal_summary)


In [ ]:
feature_n_insights = f"""
`{feature_n_name}` reflects the typical basket value of a customer and complements total spend and order frequency.
The correlation table shows that the available numerical features all have useful relationships with the target, with order count and spend-related measures providing the strongest signal.
This confirms that the provided customer-level features are appropriate starting predictors for the classification model.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<put feature name here\>`

> You can add more cells related to other feeatures in this section

In [ ]:
max_order_value_summary = train_df[['max_order_value', target_name]].copy()
max_order_value_summary['value_band'] = pd.qcut(max_order_value_summary['max_order_value'], q=10, duplicates='drop')
max_order_value_summary = max_order_value_summary.groupby('value_band').agg(
    customers=(target_name, 'size'),
    reorder_rate=(target_name, 'mean')
).reset_index(drop=True)

display(max_order_value_summary)


In [ ]:
alt.Chart(max_order_value_summary.reset_index()).mark_line(point=True).encode(
    x=alt.X('index:O', title='Max-order-value decile (low to high)'),
    y=alt.Y('reorder_rate:Q', title='Re-order rate'),
    tooltip=['customers', alt.Tooltip('reorder_rate:Q', format='.2%')]
).properties(
    title='Max order value vs re-order rate',
    width=500,
    height=320,
)


---
## D. Feature Selection


In [ ]:
features_list = X_train.columns.tolist()
print('Selected base features:', features_list)


In [ ]:
feature_audit = pd.DataFrame({
    'feature': features_list,
    'dtype': X_train[features_list].dtypes.astype(str).values,
    'missing_%': (X_train[features_list].isna().mean() * 100).round(2).values,
    'n_unique': [X_train[c].nunique(dropna=True) for c in features_list],
}).sort_values(['missing_%', 'n_unique'], ascending=[False, False])

display(feature_audit)


In [ ]:
feature_selection_explanations = """
The selected features are the four customer-level behavioural predictors produced in the preparation notebook:
`n_orders`, `total_spend`, `avg_order_value`, and `max_order_value`.
These variables directly capture customer frequency and monetary value, they are already aggregated to the correct customer grain,
and they provide a compact, interpretable starting point for a classification baseline.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation <put_name_here>

In [ ]:
training_df_clean = train_df.copy()
validation_df_clean = val_df.copy()
testing_df_clean = test_df.copy()

numeric_features = features_list.copy()
train_medians = training_df_clean[numeric_features].median()

missing_before = pd.DataFrame([
    {'split': split_name, 'missing_values': int(df[numeric_features].isna().sum().sum())}
    for split_name, df in [('train', training_df_clean), ('validation', validation_df_clean), ('test', testing_df_clean)]
])

for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    df[numeric_features] = df[numeric_features].fillna(train_medians)

missing_after = pd.DataFrame([
    {'split': split_name, 'missing_values': int(df[numeric_features].isna().sum().sum())}
    for split_name, df in [('train', training_df_clean), ('validation', validation_df_clean), ('test', testing_df_clean)]
])

display(missing_before.merge(missing_after, on='split', suffixes=('_before', '_after')))


In [ ]:
duplicate_summary = []
for split_name, df_name in [('train', 'training_df_clean'), ('validation', 'validation_df_clean'), ('test', 'testing_df_clean')]:
    df = globals()[df_name]
    duplicates_before = int(df.duplicated(subset=numeric_features + [target_name]).sum())
    globals()[df_name] = df.drop_duplicates(subset=numeric_features + [target_name]).reset_index(drop=True)
    duplicates_after = int(globals()[df_name].duplicated(subset=numeric_features + [target_name]).sum())
    duplicate_summary.append({
        'split': split_name,
        'duplicates_before': duplicates_before,
        'duplicates_after': duplicates_after,
        'rows_after_cleanup': len(globals()[df_name]),
    })

display(pd.DataFrame(duplicate_summary))


In [ ]:
data_cleaning_1_explanations = """
This first transformation handles two basic data-quality risks: missing values and duplicate customer rows.
Median imputation is robust for skewed spend variables and keeps the dataset usable without introducing extreme values.
Removing duplicate records prevents repeated patterns from biasing the classifier and keeps the customer-level modelling grain clean.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation <put_name_here>

In [ ]:
clip_bounds = []
outlier_actions = []

for feature in numeric_features:
    q1 = training_df_clean[feature].quantile(0.25)
    q3 = training_df_clean[feature].quantile(0.75)
    iqr = q3 - q1
    lower = max(0.0, q1 - 1.5 * iqr)
    upper = q3 + 1.5 * iqr
    clip_bounds.append({'feature': feature, 'lower_bound': lower, 'upper_bound': upper})

    for split_name, df in [('train', training_df_clean), ('validation', validation_df_clean), ('test', testing_df_clean)]:
        values_before = df[feature].copy()
        df[feature] = df[feature].clip(lower=lower, upper=upper)
        outlier_actions.append({
            'split': split_name,
            'feature': feature,
            'values_capped': int((values_before != df[feature]).sum()),
        })

clip_bounds = pd.DataFrame(clip_bounds)
outlier_actions = pd.DataFrame(outlier_actions)

display(clip_bounds)


In [ ]:
display(outlier_actions.pivot(index='feature', columns='split', values='values_capped').fillna(0).astype(int))


In [ ]:
data_cleaning_2_explanations = """
The spend-based variables are right-skewed and contain extreme values, so outlier capping reduces the influence of unusual customers without deleting observations.
The clipping thresholds are learned from the training data only and then applied consistently to validation and test splits, which keeps the process leakage-safe.
This makes later scaling and model fitting more stable while preserving the overall ranking structure of the features.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation <put_name_here>

In [ ]:
consistency_audit = []

for split_name, df in [('train', training_df_clean), ('validation', validation_df_clean), ('test', testing_df_clean)]:
    for feature in numeric_features:
        df[feature] = pd.to_numeric(df[feature], errors='coerce')
    df['n_orders'] = df['n_orders'].fillna(1).round().clip(lower=1).astype(int)
    spend_columns = ['total_spend', 'avg_order_value', 'max_order_value']
    df[spend_columns] = df[spend_columns].fillna(0).clip(lower=0)
    df['max_order_value'] = np.minimum(df['max_order_value'], df['total_spend'])
    df['avg_order_value'] = np.minimum(df['avg_order_value'], df['max_order_value'])

    consistency_audit.append({
        'split': split_name,
        'negative_values_remaining': int((df[spend_columns] < 0).sum().sum()),
        'rows_with_avg_gt_max': int((df['avg_order_value'] > df['max_order_value']).sum()),
        'rows_with_max_gt_total': int((df['max_order_value'] > df['total_spend']).sum()),
        'min_n_orders': int(df['n_orders'].min()),
    })

consistency_audit = pd.DataFrame(consistency_audit)
display(consistency_audit)


In [ ]:
skewness_summary = pd.DataFrame({
    'feature': numeric_features,
    'train_skewness': training_df_clean[numeric_features].skew().round(3).values,
}).sort_values('train_skewness', ascending=False)

display(skewness_summary)


In [ ]:
data_cleaning_3_explanations = """
This transformation enforces business-valid numeric values before feature engineering.
Order counts must be positive integers and spend-related values must be non-negative and logically consistent (`avg_order_value` cannot exceed `max_order_value`, and `max_order_value` cannot exceed `total_spend`).
Repairing these rules early prevents invalid ratios, unstable scaling, and misleading model patterns later in the notebook.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

In [ ]:
clean_training_df = training_df_clean[numeric_features + [target_name]].copy()
clean_validation_df = validation_df_clean[numeric_features + [target_name]].copy()
clean_testing_df = testing_df_clean[numeric_features + [target_name]].copy()

final_preparation_summary = pd.DataFrame([
    {
        'split': split_name,
        'rows': len(df),
        'columns': df.shape[1],
        'missing_values': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
        'positive_rate': float(df[target_name].mean()),
    }
    for split_name, df in [('train', clean_training_df), ('validation', clean_validation_df), ('test', clean_testing_df)]
])

display(final_preparation_summary)


In [ ]:
data_cleaning_n_explanations = """
The final data-preparation step verifies that every split shares the same cleaned schema, contains no missing values, and preserves the original target balance.
This creates a reliable starting point for feature engineering and prevents hidden data-quality issues from propagating into the modelling pipeline.
The result is a clean, customer-level dataset that is ready for consistent downstream transformations.
"""

print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)


---
## F. Feature Engineering

### F.1 New Feature "\<put_name_here\>"


In [ ]:
for df in [clean_training_df, clean_validation_df, clean_testing_df]:
    df['spend_per_order'] = (df['total_spend'] / df['n_orders'].replace(0, np.nan)).fillna(0)

feature_engineering_1_summary = clean_training_df[['spend_per_order', target_name]].groupby(target_name).describe().round(2)
display(feature_engineering_1_summary)


In [ ]:
alt.Chart(clean_training_df).mark_bar(opacity=0.7).encode(
    x=alt.X('spend_per_order:Q', bin=alt.Bin(maxbins=40), title='Spend per order'),
    y=alt.Y('count():Q', title='Customers'),
    color=alt.Color(f'{target_name}:N', title='Target class')
).properties(
    title='Distribution of spend_per_order (train split)',
    width=500,
    height=320,
)


In [ ]:
feature_engineering_1_explanations = """
`spend_per_order` combines customer frequency and monetary value into a single interpretable feature.
It highlights whether a customer tends to place many small orders or fewer higher-value orders, which can capture purchasing style more clearly than total spend alone.
Because it is derived from already-clean historical behaviour, it adds useful signal without introducing leakage.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "\<put_name_here\>"




In [ ]:
for df in [clean_training_df, clean_validation_df, clean_testing_df]:
    df['spend_concentration'] = (df['max_order_value'] / df['total_spend'].replace(0, np.nan)).fillna(0).clip(0, 1)

feature_engineering_2_summary = clean_training_df[['spend_concentration', target_name]].groupby(target_name).describe().round(3)
display(feature_engineering_2_summary)


In [ ]:
feature_engineering_2_explanations = """
`spend_concentration` measures how much of a customer's total historical spend came from their single largest order.
This helps distinguish customers with steady ordering behaviour from customers whose spend is driven by one-off purchases.
That distinction can be useful because repeat behaviour is often stronger when spending is distributed across multiple orders.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "\<put_name_here\>"

> Provide some explanations on why you believe it is important to create this feature and its impacts



In [ ]:
for df in [clean_training_df, clean_validation_df, clean_testing_df]:
    df['log_total_spend'] = np.log1p(df['total_spend'])
    df['log_avg_order_value'] = np.log1p(df['avg_order_value'])
    df['log_max_order_value'] = np.log1p(df['max_order_value'])

log_feature_skewness = pd.DataFrame({
    'raw_feature': ['total_spend', 'avg_order_value', 'max_order_value'],
    'raw_skewness': clean_training_df[['total_spend', 'avg_order_value', 'max_order_value']].skew().round(3).values,
    'log_feature': ['log_total_spend', 'log_avg_order_value', 'log_max_order_value'],
    'log_skewness': clean_training_df[['log_total_spend', 'log_avg_order_value', 'log_max_order_value']].skew().round(3).values,
})

display(log_feature_skewness)


In [ ]:
display(clean_training_df[['log_total_spend', 'log_avg_order_value', 'log_max_order_value']].head())


In [ ]:
feature_engineering_n_explanations = """
Log-transformed spend features reduce the skewness of monetary variables while preserving customer ranking.
This is useful for linear models such as logistic regression because it makes the relationship between spend and the target smoother and less dominated by very large orders.
Adding both raw and log views of spend gives the model a richer yet still interpretable representation of customer value.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

In [ ]:
engineered_features_list = [column for column in clean_training_df.columns if column != target_name]
print(f'Engineered feature count: {len(engineered_features_list)}')
print(engineered_features_list)


In [ ]:
engineered_feature_audit = pd.DataFrame({
    'feature': engineered_features_list,
    'missing_%': (clean_training_df[engineered_features_list].isna().mean() * 100).round(2).values,
    'dtype': clean_training_df[engineered_features_list].dtypes.astype(str).values,
}).sort_values(['missing_%', 'feature'], ascending=[False, True])

display(engineered_feature_audit)


---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
training_df_eng = clean_training_df.copy()
validation_df_eng = clean_validation_df.copy()
testing_df_eng = clean_testing_df.copy()

model_features = [column for column in training_df_eng.columns if column != target_name]

X_train_model = training_df_eng[model_features].copy()
y_train_model = training_df_eng[target_name].copy()

X_val_model = validation_df_eng[model_features].copy()
y_val_model = validation_df_eng[target_name].copy()

X_test_model = testing_df_eng[model_features].copy()
y_test_model = testing_df_eng[target_name].copy()

print('Prepared modelling splits:')
for split_name, X_split, y_split in [
    ('train', X_train_model, y_train_model),
    ('validation', X_val_model, y_val_model),
    ('test', X_test_model, y_test_model),
]:
    print(f"  {split_name:<10} X={X_split.shape}  y={y_split.shape}")


In [ ]:
split_audit = pd.DataFrame([
    {
        'split': split_name,
        'rows': len(X_split),
        'features': X_split.shape[1],
        'positive_rate': float(y_split.mean()),
    }
    for split_name, X_split, y_split in [
        ('train', X_train_model, y_train_model),
        ('validation', X_val_model, y_val_model),
        ('test', X_test_model, y_test_model),
    ]
])

display(split_audit)
display(X_train_model.head())


In [ ]:
data_splitting_explanations = """
The dataset was already split into training, validation, and test sets in the preparation stage, so this notebook keeps those boundaries unchanged.
That is the safest strategy because it prevents accidental leakage and allows model decisions to be tuned on validation data before the final check on the test set.
The split proportions and positive-class rates are also stable, which supports fair performance comparison across stages.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "\<put_name_here\>"

In [ ]:
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor.fit(X_train_model)
print('Fitted numeric preprocessing pipeline on the training split only.')


In [ ]:
train_prepared_preview = pd.DataFrame(
    preprocessor.transform(X_train_model),
    columns=model_features,
    index=X_train_model.index,
)

display(train_prepared_preview.head())
display(train_prepared_preview.agg(['mean', 'std']).round(3))


In [ ]:
data_transformation_1_explanations = """
Median imputation and standard scaling are fitted on the training split only, then reused for validation and test data.
This keeps the workflow leakage-safe and makes every numeric feature comparable on a common scale, which is especially important for logistic regression.
The transformation also ensures that any remaining small data issues are handled consistently during scoring.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation "\<put_name_here\>"

In [ ]:
X_train_prepared = pd.DataFrame(
    preprocessor.transform(X_train_model),
    columns=model_features,
    index=X_train_model.index,
)
X_val_prepared = pd.DataFrame(
    preprocessor.transform(X_val_model),
    columns=model_features,
    index=X_val_model.index,
)
X_test_prepared = pd.DataFrame(
    preprocessor.transform(X_test_model),
    columns=model_features,
    index=X_test_model.index,
)

print('Created prepared feature matrices for all splits.')


In [ ]:
prepared_shape_summary = pd.DataFrame([
    {'split': 'train', 'rows': X_train_prepared.shape[0], 'columns': X_train_prepared.shape[1]},
    {'split': 'validation', 'rows': X_val_prepared.shape[0], 'columns': X_val_prepared.shape[1]},
    {'split': 'test', 'rows': X_test_prepared.shape[0], 'columns': X_test_prepared.shape[1]},
])

display(prepared_shape_summary)
display(X_train_prepared.head())


In [ ]:
data_transformation_2_explanations = """
This step applies the fitted preprocessing pipeline to each split and converts the result back into labelled pandas DataFrames.
Keeping column names after transformation makes debugging, interpretation, and coefficient analysis much easier than working with anonymous arrays.
It also guarantees that training, validation, and test sets share the exact same model input schema.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation "\<put_name_here\>"

In [ ]:
prepared_data_checks = pd.DataFrame([
    {
        'split': split_name,
        'missing_values': int(df.isna().sum().sum()),
        'all_numeric': bool(all(pd.api.types.is_numeric_dtype(dtype) for dtype in df.dtypes)),
        'column_match_train': list(df.columns) == list(X_train_prepared.columns),
    }
    for split_name, df in [('train', X_train_prepared), ('validation', X_val_prepared), ('test', X_test_prepared)]
])

display(prepared_data_checks)


In [ ]:
display(X_train_prepared.describe().round(3))


In [ ]:
data_transformation_3_explanations = """
The final readiness check confirms that the prepared datasets are numeric, aligned, and free of missing values.
These are essential preconditions for stable model training and reproducible scoring.
By verifying them explicitly, the notebook reduces the risk of subtle schema or preprocessing bugs before the modelling step begins.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

In [ ]:
prepared_target_balance = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'target_rate': [y_train_model.mean(), y_val_model.mean(), y_test_model.mean()],
})

display(prepared_target_balance)


In [ ]:
print('Data preparation for modelling is complete and the notebook is ready for classifier training.')


---
## H. Save Datasets

> Do not change this code

In [ ]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

## J. Train Machine Learning Model

### J.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
algorithm_selection_explanations = """
Logistic Regression is a strong fit for this notebook because the feature set is compact, numeric, and business-friendly.
It performs well on binary classification problems, works naturally with scaled variables, and produces interpretable coefficients that can be explained to stakeholders.
Using `class_weight='balanced'` also helps the model respond to the moderate target imbalance without requiring a more complex algorithm.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
logreg_params = {
    'C': 1.0,
    'class_weight': 'balanced',
    'max_iter': 1000,
    'random_state': 42,
    'solver': 'lbfgs',
}

model = LogisticRegression(**logreg_params)
model


In [ ]:
hyperparameters_selection_explanations = """
The chosen hyperparameters aim for a robust, stable baseline.
`class_weight='balanced'` compensates for the lower share of positive cases, `max_iter=1000` ensures convergence after scaling and feature expansion,
and `C=1.0` keeps regularisation at a reasonable default so the model remains flexible without becoming overly complex.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
model.fit(X_train_prepared, y_train_model)

y_train_pred = model.predict(X_train_prepared)
y_val_pred = model.predict(X_val_prepared)
y_test_pred = model.predict(X_test_prepared)

y_train_proba = model.predict_proba(X_train_prepared)[:, 1]
y_val_proba = model.predict_proba(X_val_prepared)[:, 1]
y_test_proba = model.predict_proba(X_test_prepared)[:, 1]

model_coefficients = pd.DataFrame({
    'feature': X_train_prepared.columns,
    'coefficient': model.coef_[0],
    'abs_coefficient': np.abs(model.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)

print('Model fitted successfully.')
display(model_coefficients)


### J.4 Model Technical Performance

> Provide some explanations on model performance


In [ ]:
def metric_row(split_name, y_true, y_pred, y_proba):
    return {
        'split': split_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }

metrics_df = pd.DataFrame([
    metric_row('train', y_train_model, y_train_pred, y_train_proba),
    metric_row('validation', y_val_model, y_val_pred, y_val_proba),
    metric_row('test', y_test_model, y_test_pred, y_test_proba),
])

display(metrics_df.round(4))

print('Validation confusion matrix:')
display(pd.DataFrame(
    confusion_matrix(y_val_model, y_val_pred),
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1'],
))

print('Test confusion matrix:')
display(pd.DataFrame(
    confusion_matrix(y_test_model, y_test_pred),
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1'],
))

print('Validation classification report:')
print(classification_report(y_val_model, y_val_pred, digits=4, zero_division=0))


In [ ]:
val_metrics = metrics_df.set_index('split').loc['validation']
test_metrics = metrics_df.set_index('split').loc['test']

model_performance_explanations = f"""
The logistic regression model performs strongly on both validation and test data.
On the validation split it achieves ROC-AUC={val_metrics['roc_auc']:.3f}, F1={val_metrics['f1']:.3f}, precision={val_metrics['precision']:.3f}, and recall={val_metrics['recall']:.3f}.
The test results remain similarly strong (ROC-AUC={test_metrics['roc_auc']:.3f}, F1={test_metrics['f1']:.3f}), which suggests the workflow generalises well and is not simply overfitting the training set.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [ ]:
business_ranking = pd.DataFrame({
    'actual_reorder': y_test_model,
    'predicted_probability': y_test_proba,
}).sort_values('predicted_probability', ascending=False).reset_index(drop=True)

top_decile_n = max(1, int(len(business_ranking) * 0.10))
top_decile = business_ranking.head(top_decile_n)

top_decile_summary = pd.DataFrame({
    'metric': ['customers_in_top_10pct', 'actual_reorders_captured', 'precision_in_top_10pct', 'recall_in_top_10pct'],
    'value': [
        top_decile_n,
        int(top_decile['actual_reorder'].sum()),
        float(top_decile['actual_reorder'].mean()),
        float(top_decile['actual_reorder'].sum() / business_ranking['actual_reorder'].sum()),
    ],
})

display(top_decile_summary)
display(business_ranking.head(10))


In [ ]:
business_impacts_explanations = f"""
From a business perspective, the model is useful because it can rank customers by re-order likelihood rather than only produce hard class labels.
In the test split, the top 10% highest-scored customers contain {int(top_decile['actual_reorder'].sum())} true re-orders with a precision of {top_decile['actual_reorder'].mean():.2%}.
That means the business can focus retention campaigns on a relatively small segment while still capturing a meaningful share of likely repeat customers; the main risk is false positives, which would spend campaign budget on customers who do not return.
"""


In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [ ]:
if val_metrics['roc_auc'] >= 0.90 and val_metrics['f1'] >= 0.65:
    experiment_outcome = 'Hypothesis Confirmed'
elif val_metrics['roc_auc'] >= 0.75:
    experiment_outcome = 'Hypothesis Partially Confirmed'
else:
    experiment_outcome = 'Hypothesis Rejected'


In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
experiment_results_explanations = f"""
The experiment shows that customer-level behavioural features are sufficient to build a strong first classification model for repeat purchase prediction.
The best-performing signals came from order frequency and spend behaviour, and the final logistic regression generalised well from validation to test data.
Recommended next steps are: (1) tune the decision threshold for the business cost of false positives vs false negatives, (2) compare this interpretable baseline with tree-based models such as Random Forest or XGBoost, and (3) enrich the feature set with product-mix or recency information if more detailed transaction history becomes available.
"""


In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)

In [ ]:
final_results_table = metrics_df.copy().round(4)
display(final_results_table)


In [ ]:
print('Notebook review complete: all key classification cells now contain executable code and explanatory outputs.')
